# Demo 3 · Max-Cut with spiking neurons

Instructor demo. Spiking neurons with noise search for the best way to split a graph into two groups so that as many edges as possible run between the groups (Max-Cut). Shows that SpiNNaker2 can do more than run neural networks: optimisation problems map onto networks of neurons as well. Runs on the chip only (`qubo_neuron` is not available in Brian2); one run per graph, about one minute each.

Adapted from the py-spinnaker2 example "QUBO Max-Cut" (SpiNNcloud, Apache-2.0), with one fix: the 8-node graph file already holds the QUBO matrix and is used directly (the original example converted it a second time, which set its diagonal to zero).

## 1. The idea

- One neuron per node of the graph. A neuron that spikes in a time step puts its node into group 1 for that step, otherwise the node is in group 0.
- The QUBO matrix $Q$ becomes the connection weights: neighbours in the graph inhibit each other, so they prefer different groups.
- The `qubo_neuron` adds random noise to its membrane potential, so the network keeps jumping between configurations instead of getting stuck, a form of annealing.
- For every time step, the spike pattern is read as a split of the graph, and the number of cut edges is counted. The best split found is the result.

For a split $x \in \{0, 1\}^n$, the number of cut edges is $-x^\top Q\, x$ for the matrices used here.

In [ ]:
import itertools

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from spinnaker2 import hardware, helpers, snn


def cut_value(Q, x):
    """Number of cut edges for the split x (0/1 per node)."""
    x = np.asarray(x, dtype=float)
    return -x @ Q @ x


def best_cut(Q):
    """Exact optimum by trying all 2^n splits (fine for small graphs)."""
    n = Q.shape[0]
    return max(cut_value(Q, x) for x in itertools.product([0, 1], repeat=n))


def run_maxcut(Q, params, steps=300):
    """Run the QUBO network on one board; return the cut value per time step and the best split."""
    idx = np.nonzero(Q)
    conns = np.column_stack((idx[0], idx[1], Q[idx], np.zeros_like(Q[idx])))  # pre, post, weight, delay
    pop = snn.Population(size=Q.shape[0], neuron_model="qubo_neuron", params=params, record=["spikes"])
    net = snn.Network()
    net.add(pop, snn.Projection(pre=pop, post=pop, connections=conns))
    hardware.SpiNNcloud48NodeBoard().run(net, steps)
    states = helpers.spike_times_to_matrix(pop.get_spikes(), n_timesteps=steps)  # nodes x steps
    cuts = np.array([cut_value(Q, states[:, t]) for t in range(steps)])
    return cuts, states[:, int(np.argmax(cuts))]


def show(Q, cuts, solution, title):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 5))
    a1.plot(cuts, ".", ms=3)
    a1.axhline(best_cut(Q), ls="--", c="crimson", label=f"optimum ({best_cut(Q):.0f})")
    a1.set(xlabel="time step", ylabel="cut edges", title=f"{title}: cut found per time step")
    a1.legend(loc="lower right")
    A = (Q != 0) & ~np.eye(len(Q), dtype=bool)
    g = nx.from_numpy_array(A.astype(int))
    pos = nx.circular_layout(g)
    colors = ["#00afef" if s else "#00305C" for s in solution]
    cut = [(i, j) for i, j in g.edges if solution[i] != solution[j]]
    nx.draw_networkx_nodes(g, pos, node_color=colors, ax=a2)
    nx.draw_networkx_labels(g, pos, font_color="white", ax=a2)
    nx.draw_networkx_edges(g, pos, edgelist=[e for e in g.edges if e not in cut], style="dashed", edge_color="0.6", ax=a2)
    nx.draw_networkx_edges(g, pos, edgelist=cut, width=2.5, edge_color="crimson", ax=a2)
    a2.set_title(f"best split: {len(cut)} cut edges (red)")
    a2.axis("off")
    plt.tight_layout()
    plt.show()

## 2. A graph with 5 nodes

In [ ]:
Q5 = np.load("data/toy5_Q.npy")
print(Q5)
cuts5, best5 = run_maxcut(Q5, {"threshold": 0.1, "alpha_decay": 0.95, "reset": "reset_to_v_reset", "v_reset": 0.0})
show(Q5, cuts5, best5, "5 nodes")
print(f"best cut found: {cuts5.max():.0f}, optimum: {best_cut(Q5):.0f}, first reached at step {int(np.argmax(cuts5))}")

## 3. A graph with 8 nodes

In [ ]:
Q8 = np.load("data/toy8_Q.npy")
cuts8, best8 = run_maxcut(Q8, {"threshold": 0.15, "alpha_decay": 0.5, "reset": "reset_to_v_reset", "v_reset": 0.0})
show(Q8, cuts8, best8, "8 nodes")
print(f"best cut found: {cuts8.max():.0f}, optimum: {best_cut(Q8):.0f}, first reached at step {int(np.argmax(cuts8))}")

## 4. What this shows, and what it doesn't

- Optimisation problems can be written as networks of noisy spiking neurons; the chip then searches by itself.
- For graphs this small, a laptop finds the optimum instantly by trying all splits, as `best_cut` does above.
- Larger graphs need weights beyond the neuron's range of ±15 and careful tuning; in the original example, the 50- and 800-node graphs gave results no better than random splits. Published work on SpiNNaker2 (NeuroSA, Chen et al. 2025) reports solutions close to the state of the art at low power, but slower than a CPU in absolute time.